# Threshold Ellipses from Pre-computed Tables

Reproduces **Figure 2C** of Hong et al. (2025) — a 7 × 7 grid of threshold ellipses
in the 2-D model (W-space) for the isoluminant colour plane — by reading the
pre-computed covariance matrices from the OSF dataset and calling into the paper
repository for colour mapping.

No model fitting or JAX computation is required; the threshold covariance matrices
are read directly from the CSV tables provided on OSF.

## Subject mapping (sub number → initials)

| sub | initials |
|-----|----------|
| 1   | CH       |
| 2   | ME       |
| 4   | SG       |
| 6   | DK       |
| 7   | BH       |
| 8   | FM       |
| 10  | HG       |
| 11  | FW       |

## Before running

Download the required data files once:

```bash
python src/hong_etal_2025/download_data.py
```

Install the paper repository into this environment (if not already done):

```bash
pip install -e /path/to/ellipsoids_eLife2025
```

## 1. Setup

In [ ]:
# ruff: noqa: E402
import ast
import pathlib
import warnings

# ── Paper-repo imports ────────────────────────────────────────────────────────
# ellipsoids_eLife2025 must be installed:  pip install -e /path/to/ellipsoids_eLife2025
import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

jax.config.update("jax_enable_x64", True)  # required by the paper repo

from analysis.color_thres import color_thresholds

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = pathlib.Path("../..")
DATA_DIR = REPO_ROOT / "data"

# Subject to plot
SUB_N = 1  # 1=CH, 2=ME, 4=SG, 6=DK, 7=BH, 8=FM, 10=HG, 11=FW
SUB_INITIALS = {
    1: "CH",
    2: "ME",
    4: "SG",
    6: "DK",
    7: "BH",
    8: "FM",
    10: "HG",
    11: "FW",
}
PLANE = "Isoluminant plane"

## 2. Load threshold-ellipse covariance matrices

The file `Thres_ellipses_sub{N}.csv` contains one row per reference location on the
7 × 7 grid.  Each row stores:

- **`grid_ref`** — the 2-D W-space coordinates of the reference stimulus,
  as a comma-separated string `"x,y"`.
- **`Sigmas_thres_grid_org`** — the 2 × 2 threshold covariance matrix
  (the boundary of this ellipse is the 66.7 % correct discrimination contour),
  serialised as `"[[s11,s12],[s21,s22]]"`.  Bootstrap replicates occupy the
  remaining columns (not used here).

In [ ]:
thres_csv = (
    DATA_DIR
    / "Organized data and model predictions"
    / f"sub{SUB_N}"
    / f"Thres_ellipses_sub{SUB_N}.csv"
)

df = pd.read_csv(thres_csv)
print(f"Loaded {len(df)} rows from {thres_csv.name}")

# Parse grid reference coordinates: "x,y" → shape (49, 2)
grid_flat = np.array(
    [list(map(float, s.split(","))) for s in df["grid_ref"]]
)  # (49, 2)

# Parse threshold covariance matrices: "[[s11,s12],[s21,s22]]" → shape (49, 2, 2)
sigmas_flat = np.array(
    [ast.literal_eval(s) for s in df["Sigmas_thres_grid_org"]]
)  # (49, 2, 2)

# Reshape into 7 × 7 grids
num_pts = int(round(len(grid_flat) ** 0.5))  # 7
grid = grid_flat.reshape(num_pts, num_pts, 2)  # (7, 7, 2)
Sigmas_thres = sigmas_flat.reshape(num_pts, num_pts, 2, 2)  # (7, 7, 2, 2)

print(f"Grid shape:   {grid.shape}")
print(f"Sigmas shape: {Sigmas_thres.shape}")
print(f"Grid extents: {grid[:, :, 0].min():.2f} – {grid[:, :, 0].max():.2f}  (dim 1)")
print(f"              {grid[:, :, 1].min():.2f} – {grid[:, :, 1].max():.2f}  (dim 2)")

## 3. Colour mapping

Each ellipse is coloured to approximately match the appearance of its reference
stimulus.  `color_thresholds.W2D_to_rgb()` converts a 2-D W-space coordinate to a
display RGB triplet via the monitor calibration matrices.

In [ ]:
# Instantiate and load the colour-transformation object.
# _find_exact_path walks DATA_DIR recursively, so the files in
# data/Calibration and transformation/Transformation matrices/  are found automatically.
color_thres = color_thresholds(2, str(DATA_DIR.resolve()), PLANE)

with warnings.catch_warnings():
    warnings.simplefilter(
        "ignore"
    )  # suppress expected out-of-gamut warnings at grid edges
    color_thres.load_transformation_matrix(file_date="02242025")

print("Transformation matrices loaded.")

# Quick sanity check: convert the centre point (should be close to the display white)
rgb_centre = color_thres.W2D_to_rgb(np.array([0.0, 0.0])).squeeze()
print(f"RGB at W-space origin: {rgb_centre.round(3)}")

## 4. Reconstruct ellipse outlines

Each 2 × 2 threshold covariance matrix $\boldsymbol{\Sigma}$ defines an ellipse:

$$\mathcal{E} = \{\mathbf{c} + \mathbf{L}\mathbf{u} \;:\; \|\mathbf{u}\| = 1\}$$

where $\mathbf{L}$ is the Cholesky factor ($\mathbf{L}\mathbf{L}^\top = \boldsymbol{\Sigma}$)
and $\mathbf{c}$ is the reference location.  This traces the boundary of the set
$\{\mathbf{x} : (\mathbf{x}-\mathbf{c})^\top \boldsymbol{\Sigma}^{-1}(\mathbf{x}-\mathbf{c}) \leq 1\}$,
i.e. the 66.7 % correct discrimination contour.

In [ ]:
N_THETA = 200  # points per ellipse
theta = np.linspace(0, 2 * np.pi, N_THETA)
unit_circle = np.vstack([np.cos(theta), np.sin(theta)])  # (2, N_THETA)

# Pre-compute (ellipse_xy, color) for every grid point
ellipses = []  # list of (xy_array, rgb)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # suppress out-of-gamut warnings at grid edges
    for idx in np.ndindex(num_pts, num_pts):
        center = grid[idx]  # (2,)
        Sigma = Sigmas_thres[idx]  # (2, 2)
        L = np.linalg.cholesky(Sigma)  # lower-triangular Cholesky factor
        xy = center[:, None] + L @ unit_circle  # (2, N_THETA)
        rgb = color_thres.W2D_to_rgb(center).squeeze()  # (3,)
        ellipses.append((xy, rgb))

print(f"Computed {len(ellipses)} ellipses.")

## 5. Figure 2C — threshold ellipses on the 7 × 7 grid

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), dpi=150)

for xy, rgb in ellipses:
    ax.plot(xy[0], xy[1], color=rgb, lw=1.2)

# Axis formatting to match the paper
ticks = np.linspace(-0.7, 0.7, 5)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xlim(-0.85, 0.85)
ax.set_ylim(-0.85, 0.85)
ax.set_aspect("equal")
ax.set_xlabel("W dim 1", fontsize=9)
ax.set_ylabel("W dim 2", fontsize=9)
ax.set_title(
    f"Threshold ellipses — sub{SUB_N} ({SUB_INITIALS[SUB_N]}) — {PLANE}",
    fontsize=10,
)

plt.tight_layout()
plt.show()